# FemurFix v3 - YOLO11s @960, hip/leg-weighted fine-tune

Starts from `best_v2.pt` (YOLO11s @960, FracAtlas). v2 matched v1 on FracAtlas but
generalised better to external high-resolution femur films (OrthoFrac-XR: proximal recall
38% vs 21%), while FracAtlas hip positives are only 51 of 574 training positives.

v3 oversamples fractured hip images 6x and fractured leg images 2x in the training split
only (validation/test stay untouched), lowers the learning rate, removes rotation
augmentation, and fine-tunes for 45 epochs. Evaluate on the untouched FracAtlas test split
and on OrthoFrac-XR before replacing the deployed v1 model.

In [ ]:
import subprocess, sys, torch
print("Python:", sys.version.split()[0])
print("Torch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("WARNING: no GPU; enable a T4 accelerator.")

In [ ]:
%pip install -q "ultralytics>=8.3,<9" "albumentations>=1.4.21,<2" "onnx>=1.16" "onnxslim>=0.1.31"

In [ ]:
from pathlib import Path
import csv, shutil

def find_dir(name, marker):
    direct = [
        Path(f"/kaggle/input/{name}"),
        Path(f"/kaggle/input/datasets/birwatkar/{name}"),
    ]
    for p in direct:
        if (p / marker).exists():
            return p
    for p in Path("/kaggle/input").rglob(marker):
        return p.parent
    return None

DATA_SRC = find_dir("femurfix-fracatlas-yolo", "images/train")
META_ROOT = find_dir("fracatlas-metadata", "dataset.csv")
WEIGHTS = None
for p in Path("/kaggle/input").rglob("best_v2.pt"):
    WEIGHTS = p
    break
if DATA_SRC is None or META_ROOT is None or WEIGHTS is None:
    raise FileNotFoundError(f"DATA_SRC={DATA_SRC} META_ROOT={META_ROOT} WEIGHTS={WEIGHTS}")
print("data:", DATA_SRC)
print("meta:", META_ROOT)
print("weights:", WEIGHTS)
for split in ("train", "val", "test"):
    print(" ", split,
          len(list((DATA_SRC / "images" / split).glob("*"))), "images",
          len(list((DATA_SRC / "labels" / split).glob("*.txt"))), "labels")

In [ ]:
WORK = Path("/kaggle/temp/fracatlas-v3")
if not (WORK / "images" / "train").is_dir():
    print("Copying dataset ->", WORK)
    shutil.copytree(DATA_SRC, WORK, dirs_exist_ok=True)

rows = {r["image_id"].rsplit(".", 1)[0]: r for r in csv.DictReader(open(META_ROOT / "dataset.csv"))}
HIP_COPIES = 5   # plus the original copy -> 6x
LEG_COPIES = 1   # plus the original -> 2x
added = {"hip": 0, "leg": 0}
for img in sorted((WORK / "images" / "train").glob("*")):
    stem = img.stem
    label = WORK / "labels" / "train" / f"{stem}.txt"
    if not label.is_file() or not label.read_text().strip():
        continue
    meta = rows.get(stem, {})
    if meta.get("hip") == "1":
        copies, kind = HIP_COPIES, "hip"
    elif meta.get("leg") == "1":
        copies, kind = LEG_COPIES, "leg"
    else:
        continue
    for k in range(1, copies + 1):
        new_stem = f"{stem}__{kind}{k}"
        shutil.copyfile(img, WORK / "images" / "train" / f"{new_stem}{img.suffix}")
        shutil.copyfile(label, WORK / "labels" / "train" / f"{new_stem}.txt")
        added[kind] += 1
print("duplicated:", added)
print("train images now:", len(list((WORK / "images" / "train").glob("*"))))

In [ ]:
from pathlib import Path
RUN_ROOT = Path("/kaggle/working/runs")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
DATA_YAML = RUN_ROOT / "fracatlas_v3_data.yaml"
DATA_YAML.write_text(f'''path: {WORK.resolve().as_posix()}
train: images/train
val: images/val
test: images/test

names:
  0: fracture
''')
print(DATA_YAML.read_text())

In [ ]:
from ultralytics import YOLO

model = YOLO(str(WEIGHTS))
model.train(
    data=str(DATA_YAML),
    epochs=45,
    patience=12,
    imgsz=960,
    batch=8,
    device=0,
    workers=4,
    project=str(RUN_ROOT),
    name="fracatlas_yolo11s_960_hipweighted",
    exist_ok=True,
    seed=0,
    deterministic=True,
    optimizer="AdamW",
    lr0=0.0007,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=2.0,
    weight_decay=0.0005,
    amp=True,
    cache=True,
    plots=True,
    save=True,
    val=True,
    close_mosaic=8,
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.4,
    translate=0.1,
    scale=0.5,
    fliplr=0.5,
    flipud=0.0,
    degrees=0.0,
    mosaic=1.0,
    mixup=0.0,
    copy_paste=0.0,
    erasing=0.0,
    auto_augment=None,
)
print("best:", model.trainer.best)

In [ ]:
import json
from pathlib import Path
from ultralytics import YOLO

run_dir = Path("/kaggle/working/runs/fracatlas_yolo11s_960_hipweighted")
best = Path(run_dir / "weights" / "best.pt")
assert best.is_file(), best
metrics = YOLO(str(best)).val(
    data=str(DATA_YAML), split="test", imgsz=960, batch=8, device=0, workers=4,
    project=str(RUN_ROOT), name="fracatlas_yolo11s_960_hipweighted_test", exist_ok=True, plots=True,
)
summary = {
    "test": {
        "precision": float(metrics.box.mp),
        "recall": float(metrics.box.mr),
        "mAP50": float(metrics.box.map50),
        "mAP50-95": float(metrics.box.map),
    },
    "config": {
        "base": "best_v2.pt", "imgsz": 960, "epochs": 45, "batch": 8,
        "lr0": 0.0007, "cos_lr": True, "hip_copies": 6, "leg_copies": 2,
        "fliplr": 0.5, "degrees": 0.0,
    },
}
Path("/kaggle/working/run_summary_v3.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))

In [ ]:
# Per-anatomy image-level recall on the untouched test split, at the app gate 0.15.
import csv
from pathlib import Path
from ultralytics import YOLO

best = Path("/kaggle/working/runs/fracatlas_yolo11s_960_hipweighted/weights/best.pt")
model = YOLO(str(best))
meta = {r["image_id"].rsplit(".", 1)[0]: r for r in csv.DictReader(open(META_ROOT / "dataset.csv"))}
test_root = WORK / "images" / "test"
label_root = WORK / "labels" / "test"
groups = {"hip": [0, 0], "leg": [0, 0], "hand": [0, 0], "other": [0, 0]}
neg = [0, 0]
for img in sorted(p for p in test_root.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"}):
    lbl = label_root / f"{img.stem}.txt"
    positive = lbl.is_file() and bool(lbl.read_text().strip())
    res = model.predict(str(img), imgsz=960, conf=0.15, verbose=False)[0]
    hit = len(res.boxes) > 0
    if not positive:
        neg[0] += int(hit)
        neg[1] += 1
        continue
    m = meta.get(img.stem, {})
    if m.get("hip") == "1":
        key = "hip"
    elif m.get("leg") == "1":
        key = "leg"
    elif m.get("hand") == "1":
        key = "hand"
    else:
        key = "other"
    groups[key][0] += int(hit)
    groups[key][1] += 1
for k, (h, n) in groups.items():
    print(f"{k:5s} recall {h}/{n} = {100*h/n if n else 0:.0f}%")
print(f"negative images flagged: {neg[0]}/{neg[1]} = {100*neg[0]/neg[1] if neg[1] else 0:.0f}%")

In [ ]:
from pathlib import Path
from ultralytics import YOLO
best = Path("/kaggle/working/runs/fracatlas_yolo11s_960_hipweighted/weights/best.pt")
onnx_path = Path(YOLO(str(best)).export(format="onnx", imgsz=960, opset=12, simplify=True, dynamic=False))
Path("/kaggle/working/femur_fracture_v3.onnx").write_bytes(onnx_path.read_bytes())
Path("/kaggle/working/best_v3.pt").write_bytes(best.read_bytes())
print("exported", Path("/kaggle/working/femur_fracture_v3.onnx").stat().st_size, "bytes")

## Outputs
- `femur_fracture_v3.onnx`, `best_v3.pt`
- `run_summary_v3.json`, per-anatomy recall
- `runs/fracatlas_yolo11s_960_hipweighted/` curves and metrics

Do not deploy unless v3 beats v1 on FracAtlas test and matches or beats v2 on OrthoFrac-XR
femur images and the user sample set.